In [ ]:
from preamble import *

# Broadband LC Model

In [ ]:
def lnlike_broadbandLC(parameters, rp = 0.044, systematics = 1.0, bphase = 0, plot=False, savefigs=False, samples=None):
    """
    Log-probability function for MCMC.

    Parameters:
    - parameters: np.ndarray, current parameter values.

    Returns:
    - ln_like: float, log-likelihood of the current parameter set.
    """

    'Set the initial ln_like to zero'
    ln_like = 0.0

    'Map parameters to names for easier reference'
    params = {name: value for name, value in zip(params_config.keys(), parameters)}

    'Specify and bin the spectra for these temperatures'
    kwargs = dict(
    bin_specification = 'edges',
    bins=broadband_bin_edges,
    min=broadband_bin_edges.min(),
    max=broadband_bin_edges.max(), log=False)
    Cool, Phot = [
        bin_spectrum(
            get_BTSettl_spectrum(T=Temp), **kwargs
        )
        for Temp in [params.get('T_spot', default_params['T_spot']), params.get('T_phot', default_params['T_phot'])] ]

    'Update Planet Parameters'
    planet_parameters['u1'] = params.get('u1', default_params['u1'])
    planet_parameters['u2'] = params.get('u2', default_params['u2'])
    planet_parameters['rp'] = rp
    
    'Update stellar parameters'
    active_star.phot = Phot.flux.value
    active_star.wavelength = Phot.wavelength.to_value(u.m)
    active_star.spectrum = jnp.array([Cool.flux.value] * (n_spots+n_fixedspots))

    'Calculate fleck model'
    lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)
    if visit == 'F21':
        lc_index=2
    if visit == 'S22':
        lc_index=0
    fleck_lc = lc.T[lc_index]
    fleck_lc /= np.nanmean(fleck_lc)

    'Normalize the model and data'
    model = systematics * fleck_lc
    normalized_model = model / np.nanmean(model)
    normalized_flux = broadband_data_flux / np.nanmean(broadband_data_flux)
    'Calculate ln_like for the light curve model'
    lc_chisq = np.nansum(((normalized_flux - normalized_model) ** 2) / (broadband_relative_err ** 2))
    lc_err_weight = np.nansum(1.0 / np.sqrt(2.0 * np.pi * (broadband_relative_err)))
    ln_like += (lc_err_weight - 0.5 * lc_chisq)

    if plot:
        'Set up the data + model plots'
        fig, axes = plt.subplots(4, 1, figsize=(5, 6))
        plt.title(f"{visit} Broadband Flux Model")
        axes[0].set_xlabel("Time from midtransit")
        axes[0].set_ylabel("Relative Flux")
        axes[1].set_title("Systematic-corrected")
        axes[1].set_xlabel("Time from midtransit")
        axes[2].set_title("Residuals")
        axes[2].set_xlabel("Time from midtransit")
        axes[2].set_ylabel(r"$\sigma$")
        axes[3].set_title("Residuals")
        axes[3].set_xlabel("HST Phase")
        axes[3].set_ylabel(r"$\sigma$")

        # First plot (data + model)
        axes[0].errorbar(
            time_from_T0,
            normalized_flux,
            yerr=broadband_relative_err,fmt='o',ms=2,color='royalblue')
        # Second plot (data & model)/systematics
        axes[1].errorbar(
            time_from_T0,
            (normalized_flux/systematics)/np.nanmean(normalized_flux/systematics),
            yerr=broadband_relative_err,fmt='o',ms=2,color='royalblue')
        # third plot residuals
        axes[2].scatter(
            time_from_T0,
            (normalized_flux-normalized_model)/broadband_relative_err,s=2,color='royalblue')
        # fourth plot residuals in HST phase
        axes[3].scatter(
            bphase,
            (normalized_flux-normalized_model)/broadband_relative_err,s=2,color='royalblue')

        # Plot the models
        axes[0].plot(
            time_from_T0,
            normalized_model,
            color='k',zorder=100)
        axes[1].plot(
            time_from_T0,
            fleck_lc,
            color='k',zorder=100)
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_broadbandLC.png',dpi=250)
        plt.show()
        plt.clf()

        'Same Plot but Zoomed In'
        fig, axes = plt.subplots(4, 1, figsize=(5, 6))
        plt.title(f"{visit} Broadband Flux Model (Transit Only)")
        axes[0].set_title(f"{visit}")
        axes[0].set_xlabel("Time from midtransit")
        axes[0].set_ylabel("Relative Flux")
        axes[0].set_xlim(-0.09,0.09)
        axes[1].set_title("Systematic-corrected")
        axes[1].set_xlabel("Time from midtransit")
        axes[1].set_xlim(-0.09,0.09)
        axes[2].set_title("Residuals")
        axes[2].set_xlabel("Time from midtransit")
        axes[2].set_ylabel(r"$\sigma$")
        axes[2].set_xlim(-0.09,0.09)
        axes[3].set_title("Residuals")
        axes[3].set_xlabel("HST Phase")
        axes[3].set_ylabel(r"$\sigma$")
        axes[0].set_xlim(-0.086,0.086)
        axes[1].set_xlim(-0.086,0.086)
        axes[2].set_xlim(-0.086,0.086)
        axes[3].set_xlim(np.nanmin(bphase),np.nanmax(bphase))
        axes[0].set_ylim(np.nanmin(normalized_flux[( (orbit == 3) | (orbit == 4) | (orbit == 5) )])-3*np.nanmean(relative_err),3*np.nanmean(relative_err)+np.nanmax(normalized_flux[( (orbit == 3) | (orbit == 4) | (orbit == 5) )]))
        axes[0].set_ylim(np.nanmin(normalized_flux[( (orbit == 3) | (orbit == 4) | (orbit == 5) )])-3*np.nanmean(relative_err),3*np.nanmean(relative_err)+np.nanmax(normalized_flux[( (orbit == 3) | (orbit == 4) | (orbit == 5) )]))
        
        # First plot (data + model)
        axes[0].errorbar(
            time_from_T0,
            normalized_flux,
            yerr=broadband_relative_err,fmt='o',ms=2,color='royalblue')
        # Second plot (data & model)/systematics
        axes[1].errorbar(
            time_from_T0,
            (normalized_flux/systematics)/np.nanmean(normalized_flux/systematics),
            yerr=broadband_relative_err,fmt='o',ms=2,color='royalblue')
        # third plot residuals
        axes[2].scatter(
            time_from_T0,
            (normalized_flux-normalized_model)/broadband_relative_err,s=2,color='royalblue')
        # fourth plot residuals in HST phase
        axes[3].scatter(
            bphase,
            (normalized_flux-normalized_model)/broadband_relative_err,s=2,color='royalblue')

        # Plot the models
        axes[0].plot(
            time_from_T0,
            normalized_model,
            color='k',zorder=100)
        axes[1].plot(
            time_from_T0,
            fleck_lc,
            color='k',zorder=100)
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_broadbandLC_zoomedin.png',dpi=250)
        plt.show()
        plt.clf()

        dof = np.sum(~np.isnan(time_from_T0)) - (ndim-2*len(wavelengths))
        reduced_chisq = lc_chisq/dof
        
        print(f'This Models Reduced Chisq = {reduced_chisq:.3f} for {ndim-2*len(wavelengths)} parameters and {np.sum(~np.isnan(time_from_T0))} data points')

    return ln_like

# Spec LC Model

In [ ]:
def lnlike_specLC(parameters, rp = 0.044, systematics = 1, bphase = 0, plot=False, savefigs=False, samples=None):

    ln_like = 0.0

    # Map parameters to names for easier reference
    params = {name: value for name, value in zip(params_config.keys(), parameters)}

    'Set binning preferences and download model spectra'
    kwargs = dict(
    bin_specification = 'edges',
    bins=spec_bin_edges,
    min=spec_bin_edges.min(),
    max=spec_bin_edges.max(), log=False)
    Cool, Phot = [
        bin_spectrum(
            get_BTSettl_spectrum(T=Temp), **kwargs
        )
        for Temp in [params.get('T_spot', default_params['T_spot']), params.get('T_phot', default_params['T_phot'])] ]

    'Update Planet Parameters'
    planet_parameters['u1'] = np.array([params[f"u1_w{j}"] for j in range(1,len(spec_bin_edges))])
    planet_parameters['u2'] = np.array([params[f"u2_w{j}"] for j in range(1,len(spec_bin_edges))])
    planet_parameters['rp'] = rp

    'Update stellar parameters'
    active_star.phot = Phot.flux.value
    active_star.wavelength = Phot.wavelength.to_value(u.m)
    active_star.spectrum = jnp.array([Cool.flux.value] * (n_spots+n_fixedspots))

    lc, contam, X, Y, spectrum_at_transit = active_star.transit_model(**planet_parameters)        
    for i, lc_i in enumerate(lc.T):
        lc_i /= np.nanmean(lc_i)
        model_i = systematics * lc_i
        model_i /= np.nanmedian(model_i)
        normalized_flux = data_flux[i, :]/np.nanmedian(data_flux[i, :])
        chisq = np.nansum(((normalized_flux - model_i) ** 2) / (relative_err[i, :] ** 2))
        err_weight = np.nansum(1.0 / np.sqrt(2.0 * np.pi * (relative_err[i, :])))
        ln_like += (err_weight - 0.5 * chisq)
        
    if plot:
        'Plot the transmission and contamination spectra'
        'First Remove nan data'
        mask = ~np.isnan(time_from_T0)
        nonnan_time_from_T0 = time_from_T0[mask]
        active_star.times = nonnan_time_from_T0
        _, nanremoved_contam, _, _, nanremoved_spectrum_at_transit = active_star.transit_model(**planet_parameters)
        fig, axs = plt.subplots(3,1,figsize=(5,5))
        axs[0].set_title('Contaminated Transmission Spectrum')
        axs[0].scatter(wavelengths, nanremoved_contam * 1e6 ,s=2,color='k')
        axs[1].set_title('Stellar Contamination')
        axs[1].scatter(wavelengths, (nanremoved_contam-planet_parameters['rp']**2)*1e6,s=2,color='k')
        axs[2].set_title('De-contaminated Transmission Spectrum')
        axs[2].scatter(wavelengths, [(planet_parameters['rp'])**2 * 1e6]*len(wavelengths),s=2,color='k')
        axs[0].set_ylabel('Transit Depth (ppm)')
        axs[1].set_ylabel('Transit Depth (ppm)')
        axs[2].set_ylabel('Transit Depth (ppm)')
        plt.xlabel(r'Wavelength ($\mu$m)')
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_{len(spec_bin_edges)-1}bins_transmission.png',dpi=250)
        plt.show()
        plt.clf()

        'Re-set to the original array (containing nans)'
        active_star.times = time_from_T0
        
        'Set up the data + model plots'
        fig, axes = plt.subplots(4, 1, figsize=(5,12))
        axes[0].set_title(f"{visit}")
        axes[0].set_xlabel("Time from midtransit")
        axes[0].set_ylabel("Relative Flux")
        axes[1].set_title("Systematic-corrected")
        axes[1].set_xlabel("Time from midtransit")
        axes[2].set_title("Residuals")
        axes[2].set_xlabel("Time from midtransit")
        axes[2].set_ylabel(r"$\sigma$")
        axes[3].set_title("Residuals")
        axes[3].set_xlabel("HST Phase")
        axes[3].set_ylabel(r"$\sigma$")
        
        chisq_per_wavelength = []
        for i, lc_i in enumerate(lc.T):
            lc_i /= np.nanmean(lc_i)
            model_i = systematics * lc_i
            model_i /= np.nanmedian(model_i)
            normalized_flux = data_flux[i, :]/np.nanmedian(data_flux[i, :])
            chisq = np.nansum(((normalized_flux - model_i) ** 2) / (relative_err[i, :] ** 2))
            chisq_per_wavelength.append(chisq)
        
            # First plot (data + model)
            axes[0].errorbar(
                time_from_T0,
                normalized_flux - (0.002*i),
                yerr=relative_err[i, :],fmt='o',ms=1,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # Second plot (data & model)/systematics
            axes[1].errorbar(
                time_from_T0,
                (normalized_flux/systematics)/np.nanmean(normalized_flux/systematics) - (0.002*i),
                yerr=relative_err[i, :],fmt='o',ms=1,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # third plot residuals
            axes[2].scatter(
                time_from_T0,
                (normalized_flux-model_i)/relative_err[i, :],s=2,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # fourth plot residuals in HST phase
            axes[3].scatter(
                bphase,
                (normalized_flux-model_i)/relative_err[i, :],s=2,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))

            # Plot the models
            axes[0].plot(
                time_from_T0,
                model_i - (0.002*i),
                color='k',zorder=100)
            axes[1].plot(
                time_from_T0,
                lc_i - (0.002*i),
                color='k',zorder=100)
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_{len(spec_bin_edges)-1}bins_specLC.png',dpi=250)
        plt.show()
        plt.clf()

        'Set up the data + model plots'
        fig, axes = plt.subplots(4, 1, figsize=(4, 10))
        axes[0].set_title(f"{visit}")
        axes[0].set_xlabel("Time from midtransit")
        axes[0].set_ylabel("Relative Flux")
        # axes[0].set_xlim(-0.085,0.085)
        axes[1].set_title("Systematic-corrected")
        axes[1].set_xlabel("Time from midtransit")
        # axes[1].set_xlim(-0.085,0.085)
        axes[2].set_title("Residuals")
        axes[2].set_xlabel("Time from midtransit")
        axes[2].set_ylabel(r"$\sigma$")
        # axes[2].set_xlim(-0.085,0.085)
        axes[3].set_title("Residuals")
        axes[3].set_xlabel("HST Phase")
        axes[3].set_ylabel(r"$\sigma$")
        axes[0].set_xlim(-0.086,0.086)
        axes[1].set_xlim(-0.086,0.086)
        axes[2].set_xlim(-0.086,0.086)
        axes[3].set_xlim(np.nanmin(bphase),np.nanmax(bphase))
        for i, lc_i in enumerate(lc.T):
            lc_i /= np.nanmean(lc_i)
            model_i = systematics * lc_i
            model_i /= np.nanmedian(model_i)

            normalized_flux = data_flux[i, :]/np.nanmedian(data_flux[i, :])
        
            # First plot (data + model)
            axes[0].errorbar(
                time_from_T0,
                normalized_flux - (0.002*i),
                yerr=relative_err[i, :],fmt='o',ms=1,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # Second plot (data & model)/systematics
            axes[1].errorbar(
                time_from_T0,
                (normalized_flux/systematics)/np.nanmean(normalized_flux/systematics) - (0.002*i),
                yerr=relative_err[i, :],fmt='o',ms=1,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # third plot residuals
            axes[2].scatter(
                time_from_T0,
                (normalized_flux-model_i)/relative_err[i, :],s=2,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))
            # fourth plot residuals in HST phase
            axes[3].scatter(
                bphase,
                (normalized_flux-model_i)/relative_err[i, :],s=2,
                color=plt.cm.turbo((wavelengths[i] - wavelengths.min()) / np.ptp(wavelengths)))

            # Plot the models
            axes[0].plot(
                time_from_T0,
                model_i - (0.002*i),
                color='k',zorder=100)
            axes[1].plot(
                time_from_T0,
                lc_i - (0.002*i),
                color='k',zorder=100)

        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_{len(spec_bin_edges)-1}bins_zoomedspeclcs.png',dpi=250)
        plt.show()
        plt.clf()

        total_chisq = np.nansum(chisq_per_wavelength)
        dof = np.sum(~np.isnan(time_from_T0))*len(wavelengths) - (ndim-2)
        reduced_chisq = total_chisq/dof
        print(f'This Models Reduced Chisq = {reduced_chisq:.3f} for {ndim-2} parameters and {np.sum(~np.isnan(time_from_T0))*len(wavelengths)} data points')

    return ln_like

# OOT Spec model

In [ ]:
def lnlike_OOTspec(parameters, plot=False, savefigs=False, samples=None,
                   convolution_method='astropy', kernel_type='astropy'):

    ln_like = 0.0

    'Map parameters to names for easier reference'
    params = {name: value for name, value in zip(params_config.keys(), parameters)}
        
    Cool = btsettl_grid(float(params['T_spot']))
    Phot = btsettl_grid(float(params['T_phot']))
    
    'Calculate combined spectrum'
    _f = (params['f_spot']*Cool + (1.0-params['f_spot'])*Phot)
    exclude_nans = ~np.isnan(_f)    
    model_wave = btsettl_wavelengths[exclude_nans]
    model_flux = _f[exclude_nans]
    convolved = convolve_spectrum(model_wave, model_flux, sigma=filter_sigma, method = convolution_method, kernel_type = kernel_type)
    binned_model_flux = bintogrid(model_wave, convolved, newx=mean_wave)['y'] * 3.44768e-18
    model_flux = binned_model_flux * params['spec_scale_factor']
    
    'Calculate ln_like for the light curve model'
    OOT_chisq = np.nansum(((calibrated_data_flux.value - model_flux) ** 2) / ((0.005*calibrated_data_flux.value) ** 2))
    OOT_err_weight = np.nansum(1.0 / np.sqrt(2.0 * np.pi * (0.005*calibrated_data_flux.value)))
    ln_like += (OOT_err_weight - 0.5 * OOT_chisq)

    if plot:
        plt.errorbar(mean_wave, calibrated_data_flux, yerr = (0.005*calibrated_data_flux),fmt='o',ms=2)
        plt.plot(mean_wave, model_flux)
        if savefigs:
            plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_OOTSpec.png',dpi=250)
        plt.show()
        plt.clf()

        dof = np.sum(~np.isnan(mean_wave)) - 4
        reduced_chisq = OOT_chisq/dof
        
        print(f'This Models Reduced Chisq = {reduced_chisq:.3f} for {4} parameters and {np.sum(~np.isnan(mean_wave))} data points')

    return ln_like

In [ ]:
def lnprob(parameters, plot=False, savefigs=False, samples=None):
    """
    Log-probability function for MCMC.

    Parameters:
    - parameters: np.ndarray, current parameter values.
    - kwargs: dict, additional arguments required for model computation.

    Returns:
    - ln_like: float, log-likelihood of the current parameter set.
    """
    ln_like = 0.0

    # Map parameters to names for easier reference
    params = {name: value for name, value in zip(params_config.keys(), parameters)}

    'Check if all priors are satisfied'
    if all(bounds[0] <= params[key] <= bounds[1] for key, bounds in priors.items()):
        
        active_star.times = time_from_T0
        active_star.T_eff = params.get('T_phot',default_params['T_phot'])
        active_star.temperature = jnp.array([params.get("T_spot", default_params["T_spot"])] * (n_spots+n_fixedspots))
        active_star.rad = jnp.array([10**(params['log_fixedspot_radii'])] * (n_spots+n_fixedspots))
        active_star.lon = active_star.lon.at[-n_spots:].set([params.get(f"spot{i}_lon", default_params[f"spot{i}_lon"]) for i in range(1, n_spots + 1)][::-1])
        active_star.lat = active_star.lat.at[-n_spots:].set([params.get(f"spot{i}_lat", default_params[f"spot{i}_lat"]) for i in range(1, n_spots + 1)][::-1])
        active_star.rad = active_star.rad.at[-n_spots:].set([params.get(f"spot{i}_rad", default_params[f"spot{i}_rad"]) for i in range(1, n_spots + 1)][::-1])

        planet_parameters['t0'] = params.get('T_0', default_params['T_0'])
        planet_parameters['a'] = params.get('a_rstar', default_params['a_rstar'])
        planet_parameters['ecc'] = params.get('ecc', default_params['ecc'])
        planet_parameters['inclination'] = np.radians(params.get('planet_i', default_params['planet_i']))
        'Calculate Systematics'
        ramp = ramp_model(phase = ramp_phase,
                          r1=params.get('r1', default_params['r1']),
                          r2=params.get('r2', default_params['r2']),
                          r3=params.get('r3', default_params['r3']),)
        bphase = ( (img_date-ref_time[0]+0.02)/(params.get('HST_period', default_params['HST_period'])) ) % 1
        breathing = breathing_model(phase = bphase,
                                    b1=params.get('b1', default_params['b1']),
                                    b2=params.get('b2', default_params['b2']),
                                    b3=params.get('b3', default_params['b3']),
                                    b4=params.get('b4', default_params['b4']),)
        # slope = params.get('slope', default_params['slope'])
        systematics = (breathing * ramp)
        
        if plot:
            'Plot the stellar surface'
            active_star.plot_star(
                t0=planet_parameters['t0'],
                rp=np.nanmean(planet_parameters['rp']),  # Exoplanet radius in units of stellar radii
                a=planet_parameters['a'],  # Planetary semi-major axis in units of stellar radii
                inclination=planet_parameters['inclination'],  # Planetary orbital inclination [radians]
                ecc=planet_parameters['ecc'] )
            plt.title(f'{visit} Stellar Background at T0')
            if savefigs:
                plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_starmap.png', dpi=250)
            plt.show()
            plt.clf()
            
            'Plot the systematics models'
            fig, axs = plt.subplots(4,1, figsize=(6,6))
            plt.title('Systematics')
            axs[0].set_title('Ramp')
            axs[1].set_title('Breathing')
            axs[2].set_title('Breathing')
            axs[3].set_title('Ramp x Breathing')
            axs[0].scatter(ramp_phase, ramp,color='k',alpha=0.8,s=2)
            axs[1].scatter(bphase, breathing,color='k',alpha=0.8,s=2)
            axs[2].scatter(ramp_phase, breathing,color='k',alpha=0.8,s=2)
            axs[3].scatter(img_date, ramp*breathing,color='k',alpha=0.8,s=2)
            axs[0].set_xlabel('Ramp Phase')
            axs[1].set_xlabel('HST Orbital Phase')
            axs[2].set_xlabel('Ramp Phase')
            axs[3].set_xlabel('Time (BJD)')
            if savefigs:
                plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}steps_systematics_models.png',dpi=250)
            plt.show()
            plt.clf()
            
        if calculate_broadband_lc:
            ln_like += lnlike_broadbandLC(parameters,
                                          rp = params['Rp_Rs'],
                                          systematics = systematics,
                                          bphase = bphase,
                                          plot = plot, savefigs = savefigs, samples = samples)
        if calculate_spec_lc:
            ln_like += lnlike_specLC(parameters,
                                     rp = params['Rp_Rs'],
                                     systematics = systematics, 
                                     bphase=bphase,
                                     plot = plot, savefigs = savefigs, samples = samples)
        if calculate_OOT_spec:
            ln_like += lnlike_OOTspec(parameters, plot=plot, savefigs=savefigs, samples=samples)
        
        return ln_like

    else:

        return -np.inf

# Define System Parameters

In [ ]:
default_params={
    # Fixed parameters
    "log_g":4.52,
    "P_orb":8.463,
    "stellar_i":85.0,
    'P_rot':4.86,
    "Rs":0.8,
    "Ms":0.6,
    "Mp":10.0,
    "metallicity":0.12,
    "planet_i":89.5,
    "HST_period":0.066,
    # Ramp Model Parameters
    "r1": 18,
    "r2": -7,
    "r3": 0,
    #Breathing params
    "b1":0,
    "b2":0,
    "b3":0,
    "b4":0,
    # Transit Parameters
    "u1":0.22,
    "u2":0.35,
    "T_0":0.0,
    "Rp_Rs": 0.045,
    "a_rstar": 18.2,
    "T_ref":600,
    "ecc": 0.045,
    "log_fixedspot_radii":-1.6,
    "f_spot":0.2,
    "spec_scale_factor": 1.0,
    "T_spot": 3100,
    "T_phot": 4200,
    "spot1_lon":0.875,
    "spot1_lat":1.79,
    "spot1_rad":0.07,
    "spot2_lon":-1.1,
    "spot2_lat":1.2,
    "spot2_rad":0.1,
    "spot3_lon":0.09,
    "spot3_lat":1.48,
    "spot3_rad":0.32,
}

_params_config={
    # Ramp Model Parameters
    "r1": (17.0,18.5),
    "r2": (-8.0,-6.5),
    "r3": (-0.0001,0.0001),
    #Breathing params
    "b1":(-0.01,0.01),
    "b2":(-0.01,0.01),
    "b3":(-0.01,0.01),
    "b4":(-0.01,0.01),
    # Planetary Parameters
    "u1":(0.1,0.5),
    "u2":(0.1,0.5),
    # "T_0": (-0.0001,0.0001),
    "Rp_Rs": (0.038,0.043),
    "a_rstar": (18.0,18.5),
    # "ecc": (0.02,0.05),
    # Stellar Parameters
    "f_spot":(0.1,0.5),
    "spec_scale_factor": (0.8, 0.9),
    "T_spot": (3000,3300),
    "T_phot": (4000,4200),
    "log_fixedspot_radii":(-1.8,-1.5), #This is the size scale of spots at the active latitudes
    # This spot is occulted
    "spot1_lon": (0.84,0.89),
    "spot1_lat": (1.76,1.83),
    "spot1_rad": (0.05,0.07),
    #This spot controls the overall rotational modulation
    "spot2_lon": (-1.1,-0.8),
    "spot2_lat": (0.9,1.6),
    "spot2_rad": (0.15,0.25),
    "spot3_lon":(0.08,1.0),
    "spot3_lat":(1.48,1.55),
    "spot3_rad":(0.2,0.35),
}

_priors={
    # Ramp Model Parameters
    "r1": (5.0, 30.0),
    "r2": (-20, -1.0),
    "r3": (-0.01, 0.01),
    #Breathing params
    "b1":(-0.5, 0.5),
    "b2":(-0.5, 0.5),
    "b3":(-0.5, 0.5),
    "b4":(-0.5, 0.5),
    # Transit Parameters
    "u1":(0.1, 0.5),
    "u2":(0.1, 0.5),
    "Rp_Rs": (0.03,0.055),
    "a_rstar": (17.0, 21.0),
    # "ecc": (0.0, 0.2), # make zero? use impact parameter instead
    # Stellar Parameters
    "f_spot":(0.0,0.5),
    "spec_scale_factor": (0.1, 3.0),
    "T_spot": (2900,3500),
    "T_phot": (3700,4300),
    "log_fixedspot_radii":(-2.5,-1),
    #This is the spot crossing in S22
    "spot1_lon": (0.0, 0.92),
    "spot1_lat": (1.7, 1.84),
    "spot1_rad": (0.0, 0.2),
    #these spots control the rotational modulation
    "spot2_lon": (-1.5, 0.0),
    "spot2_lat": (0.9, 1.8),
    "spot2_rad": (0.0, 0.5),
    "spot3_lon":(0.03,1.2),
    "spot3_lat":(1.2,1.55),
    "spot3_rad":(0.0,0.5),
}

# Initialize Models

In [ ]:
'Initialization'
n_spots = 3
n_fixedspots = 800
nwalkers = 200
nsteps = 2000
burnin = int(0.5*nsteps)
calculate_broadband_lc = True
calculate_spec_lc = True
calculate_OOT_spec = True

'Set the binning'
orbits_to_exclude = np.array([0])
orbits_to_include = np.array([1,2,3,4,5,6,7])
F21_bin_edges = np.array([1.14064,1.1777,
                            1.21477,1.26573,
                            1.32132,1.35838,
                            1.45104,
                            1.53907,
                            1.60393,1.642])* u.micron
S22_bin_edges = np.array([0.84027,0.85502,0.87469,0.89436,
                            0.91402,0.93369,0.95336,0.97303,0.99269,
                            1.01236,1.03203,1.05170,1.07136,1.09103,
                            1.11070,1.13]) * u.micron

broadband_bin_edges = np.array([0.84027, 1.13,
                                1.14,1.642])*u.micron

'Set binning preferences and download initial spectra'
kwargs = dict(
    bin_specification = 'edges',
    bins=S22_bin_edges,
    min=S22_bin_edges.min(),
    max=S22_bin_edges.max(),
    log=False)
Cool, Phot = [
    bin_spectrum(
        get_BTSettl_spectrum(T=Temp), **kwargs)
    for Temp in [default_params['T_spot'], default_params['T_phot']] ]

'Initialize planet parameters and the active_star'
planet_parameters = dict(
    inclination = np.radians(default_params['planet_i']),
    a = default_params['a_rstar'],
    rp = 0.044,
    period = default_params['P_orb'],
    t0 = 0,
    # omega = np.radians(88.5),
    ecc = default_params['ecc'],
    u1 = 0.22,
    u2 = 0.35,)
active_star = ActiveStar(
    times=np.linspace(-0.25,0.25,300),
    inclination=np.radians(default_params['stellar_i']),  # stellar inc [rad]
    T_eff=default_params['T_phot'],
    wavelength=Phot.wavelength.to_value(u.m), #use data wavelength array
    phot=Phot.flux.value, #This is the model spectrum for the photosphere temp
    P_rot=default_params['P_rot'])

for i in range(1, n_fixedspots + 1):
    # Generate random values within the parameter space for each spot
    lon = np.random.uniform(-2.8,2.8)
    lat = random.choice( [np.random.beta(2, 1)*(0.9 - 0.1) + 0.1, np.random.beta(1,2)*(3.04 - 2.24) + 2.24 ] )
    rad = 0.01

    latspot = dict(
        lon=lon,  # Longitude [rad]
        lat=lat,  # Latitude [rad]
        rad=rad,  # Radius [R_star]
        spectrum=Cool.flux.value,  # Use provided spectrum
        temperature=default_params['T_spot'],  # Use provided temperature
    )
    # Add the spot to the active star
    active_star.add_spot(**latspot)
    
for i in [1,2,3]:
    # Generate random values within the parameter space for each spot
    lon = default_params[f"spot{i}_lon"]
    lat = default_params[f"spot{i}_lat"]
    rad = default_params[f"spot{i}_rad"]

    spot = dict(
        lon=lon,  # Longitude [rad]
        lat=lat,  # Latitude [rad]
        rad=rad,  # Radius [R_star]
        spectrum=Cool.flux.value,  # Use provided spectrum
        temperature=default_params['T_spot'],  # Use provided temperature
    )
    # Add the spot to the active star
    active_star.add_spot(**spot)

# 'Plot the stellar surface'
active_star.plot_star(
    t0=planet_parameters['t0'],
    rp=planet_parameters['rp'],  # Exoplanet radius in units of stellar radii
    a=planet_parameters['a'],  # Planetary semi-major axis in units of stellar radii
    inclination=planet_parameters['inclination'],  # Planetary orbital inclination [radians]
    ecc=planet_parameters['ecc'] )
plt.show()
plt.clf()

# Run the MCMC

In [ ]:
for visit in ['F21','S22']:
    params_config = _params_config
    priors = _priors

    if visit=='F21':
        orbits_to_exclude = np.array([0])
        orbits_to_include = np.array([1,2,3,4,5,6,7])
        spec_bin_edges = F21_bin_edges
        err_inflation = 1.0
        broadband_err_inflation = 2.0
        oot_err_inflation = 150.0
        filter_sigma = 20*u.pixel

    if visit=='S22':
        orbits_to_exclude = np.array([0,2,7])
        orbits_to_include = np.array([1,3,4,5,6])
        spec_bin_edges = S22_bin_edges
        err_inflation = 1.0
        broadband_err_inflation = 2.0
        oot_err_inflation = 150.0
        filter_sigma = 10*u.pixel

    print('Running ',visit)
    print('')
        
    'Extract exposure times and mid-transit times from the dictionary'
    predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
    exposureTime = visits[f'{visit}']['exp (s)'].value
    
    'Read in the trimmed rainbows from process_pacman_spectra.ipynb'
    broadband_data_fluxes = []
    data_fluxes = []
    rel_spec_errs = []
    img_dates = []
    wave_arrays = []
    for direction in ['Forward','Reverse']:
        rainbow = read_rainbow(f"../data/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        broadband_data_fluxes.append(np.nansum(rainbow.flux.value, axis=0))
        img_dates.append(rainbow.time.value)
        binned_rainbow = rainbow.bin(wavelength_edges=spec_bin_edges).normalize()
        wave_arrays.append(binned_rainbow.wavelength.value)
        data_fluxes.append(binned_rainbow.flux.value)
        rel_spec_errs.append(binned_rainbow.uncertainty.value)
    wavelengths = np.nanmean(wave_arrays, axis=0)
    img_date = np.nanmean(img_dates,axis=0)
    data_flux = np.nansum(data_fluxes,axis=0)/2
    relative_err = err_inflation * np.sqrt((rel_spec_errs[0]**2 + rel_spec_errs[1]**2))/np.sqrt(2)
    for i in range(len(wavelengths)):
            print(f'{visit} | {wavelengths[i]:.6f} | mean relative uncertainty = {int(np.nanmean((relative_err[i,:]))*1e6)} ppm')
    time_from_T0 = img_date - predicted_T0
    broadband_data_flux = np.nansum(broadband_data_fluxes,axis=0)
    broadband_relative_err = broadband_err_inflation*(np.sqrt(broadband_data_flux)/broadband_data_flux)
    print(f'Median Photon Error for {visit} broadband flux = {int( np.nanmedian((np.sqrt(broadband_data_flux)/broadband_data_flux))*1e6 ) } ppm')

    'Calculate the average OOT spectrum'
    exptime = visits[f'{visit}']['exp (s)']
    binwidth = visits[f'{visit}']['native resolution']
    grism = visits[f'{visit}']['Grism']
    unbinned_wavelengths = []
    summed_spectra = []
    for direction in ['Forward','Reverse']:
        trimmed_r = read_rainbow(f"../data/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        summed_spectra.append(np.nanmedian(trimmed_r.flux.value,axis=1) )
        unbinned_wavelengths.append(trimmed_r.wavelength.value)
    
    meanOOTspec = np.nanmean(summed_spectra,axis=0)
    mean_wave = np.nanmean(unbinned_wavelengths,axis=0)
    e_per_s = meanOOTspec / exptime
    e_per_s_per_angstrom = e_per_s / binwidth
    _w, _s, _e = read_sensitivity_curve(grism=grism)
    binned_filter_response = bintogrid(_w.value, _s.value, newx=mean_wave)['y'] * u.cm**2 / u.erg
    calibrated_data_flux = e_per_s_per_angstrom / binned_filter_response
    # plt.plot(mean_wave,calibrated_data_flux.value)
        
    'Label the orbits'
    orbit = np.zeros_like(img_dates[0])
    for j in range(len(img_dates[0])):
        if j >= 1:
            if (img_dates[0][j] - img_dates[0][j - 1]) > 0.01:
                orbit[j] = (orbit[j - 1] + 1)
            else:
                orbit[j] = orbit[j - 1]
    
    'Trim the first point from each orbit'
    ref_time = []
    for o in np.unique(orbit):
        first_index = np.where(orbit == o)[0][0]
        ref_time.append(img_dates[0][first_index])
        data_flux[:, first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
        broadband_data_flux[first_index] = np.nan
        relative_err[:, first_index] = np.nan
        broadband_relative_err[first_index] = np.nan
        img_date[first_index] = np.nan
        time_from_T0[first_index] = np.nan

    'Set data to nan if it was in the pre-defined list of orbits to exclude'
    for orbit_to_exclude in orbits_to_exclude:
        data_flux[:, orbit == orbit_to_exclude] = np.nan
        broadband_data_flux[orbit == orbit_to_exclude] = np.nan
        relative_err[:, orbit == orbit_to_exclude] = np.nan
        broadband_relative_err[orbit == orbit_to_exclude] = np.nan
        img_date[orbit == orbit_to_exclude] = np.nan
        time_from_T0[orbit == orbit_to_exclude] = np.nan
    
    'Populate ramp_phase time arrays'
    phase_list=[]
    for o in [0,1,2,3,4,5,6,7]:
        rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
        phase_list.append(rphase)
    ramp_phase = np.concatenate(phase_list)

    'Add wavelength-dependent u1 and u2 parameters'
    for i in range(1,len(spec_bin_edges)):
        params_config[f"u1_w{i}"] = (0.2, 0.3)
        params_config[f"u2_w{i}"] = (0.3, 0.4)
        # Add the same uniform prior for all `u1_wi` and `u2_wi`
        priors[f"u1_w{i}"] = (0.1, 0.5)
        priors[f"u2_w{i}"] = (0.1, 0.5)

    ndim = len(priors)
    variable_names = list(params_config.keys())
    p0 = initialize_walkers(nwalkers, params_config)
    
    'Select a random set of parameters to test the initial models'
    example_params = p0[np.random.randint(0, nwalkers - 1), :]
    lnprob(example_params, plot=True,savefigs=False,samples=None)
    # continue
    
    'Set up MCMC'
    label=f'{visit}_{n_spots}spots_{nsteps}steps_{int(len(spec_bin_edges)-1)}bins_mcmc_opaqueplanet'
    samples_fname = f"../data/samples/{label}.h5"
    backend = emcee.backends.HDFBackend(samples_fname)
    backend.reset(nwalkers, ndim)        
    sampler = emcee.EnsembleSampler(nwalkers, ndim, lnprob,backend=backend)

    'Run the MCMC'
    print(f'Running MCMC sampler for {visit}')
    result = sampler.run_mcmc(p0, nsteps, store=True, progress=True)
    samples = sampler.chain[:, burnin:, :].reshape((-1, ndim)).T
    
    'Check for convergence (all should be > 100)'
    for i in range(len(samples)):
        tau_f = emcee.autocorr.integrated_time(samples[i])
        print('(Nsteps-burnin)*nwalkers/tau=',int((nsteps-burnin)*nwalkers/tau_f))
    
    'Return the Most likely set of parameters'
    flat_log_prob = sampler.get_log_prob(discard=burnin, flat=True)
    best_index = np.argmax(flat_log_prob)  # Index of the highest log-probability
    highest_prob_params = samples[:, best_index]
    print("Highest Likelihood Parameters:")
    for i,param in enumerate(highest_prob_params):
        print(f'{variable_names[i]}={param:.7f}')
    
    'Extract and display best-fit parameters and uncertainties'
    median_params = []
    err_upper = []
    err_lower = []
    for i, param_samples in enumerate(samples):
        sig_1_results = np.percentile(param_samples, [15.9, 50.0, 84.1])
        median_params.append(sig_1_results[1])
        err_upper.append(sig_1_results[2] - sig_1_results[1])
        err_lower.append(sig_1_results[1] - sig_1_results[0])
    params_dict = {name: median_params[i] for i, name in enumerate(params_config.keys())}
    for i, label in enumerate(variable_names):
        print(f"{label}: {median_params[i]:.6f} +{err_upper[i]:.6f} -{err_lower[i]:.7f}")
        print("--------------")

    lnprob(highest_prob_params, plot=True, savefigs=True, samples=samples)

    'Make corner plot'
    rng = 0.9995
    fig = corner.corner( 
        samples.T,show_titles=True, labels=variable_names,
        range=[rng]*ndim,smooth=1,quantiles=(0.16, 0.5, 0.84),
        fill_contours=True, plot_datapoints=False,title_kwargs={"fontsize": 15},title_fmt='.4f',
        hist_kwargs={"linewidth": 2.5},levels=[(1-np.exp(-0.5)),(1-np.exp(-2)),(1-np.exp(-4.5))])
    plt.suptitle(f"{visit} MCMC Results")
    plt.savefig(f'../figs/{visit}_{n_spots}spots_{nsteps}nsteps_{len(spec_bin_edges)-1}bins_corner.pdf')
    plt.clf()